# Tutorial 2 — Bayesian inversion of the synthetic data set

This notebook estimates the properties of the river bed from the temperatures written by tutorial 1, and checks the result against the values used to generate them:

1. the data set of `.data/<point name>/` is read, as the data of a real point would be;
2. an `InversionColumn` (the `Column` of `pyheatmy` with the MCMC of `bayesian_inversion`) is built and the prior ranges of the parameters are set;
3. the MCMC (DREAM, several chains) samples the posterior distribution of the parameters;
4. the chains are inspected, and the first part of each chain is discarded;
5. only then is `truth.json` opened, to compare the posterior with the true parameters and the true Darcy velocity.

The data set of tutorial 1 is in `.data/`; running `01_generate_data.ipynb` writes the same files again. The MCMC takes about four minutes.

**Which parameters are estimated.** Only the intrinsic permeability $k$ and the thermal conductivity of the solid $\lambda_s$ are estimated here; the porosity $n$ and the volumetric heat capacity of the solid $\rho_s c_s$ are fixed, as if they had been measured in the laboratory, and the lateral source $q_s$ is fixed to zero. With these four parameters free, the chains still had not converged after 500 iterations (several minutes), because the temperatures constrain combinations of $n$, $\lambda_s$ and $\rho_s c_s$ rather than each of them; estimating all of them is left to a later tutorial.

In [ ]:
import json
import random
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pyheatmy import Column, Layer
from pyheatmy.config import ZERO_CELSIUS

from bayesian_inversion.inversion_column import InversionColumn

## 1. Settings

- **Data set.** The point written by tutorial 1, and what is known about it without its `truth.json`: the depths of the sensors and the accuracy of the temperature sensors.
- **Grid.** The number of cells of the direct model; the same as in tutorial 1, so that the grid adds no error of its own.
- **Priors.** A range for each estimated parameter, and a value for each fixed one. The sampling scale is chosen by `bayesian_inversion` from the range: a range over several orders of magnitude, like the one of $k$, is sampled uniformly in $\log_{10} k$.
- **MCMC.** The number of chains and of iterations, the fraction of each chain discarded before the posterior is computed, and the rule that sets aside a chain stuck far from the others (see section 4).
- **Seeds.** The MCMC of `bayesian_inversion` draws random numbers from both Python's `random` module and NumPy, so both are seeded.

In [ ]:
POINT_NAME = "Synthetic01"
DATA_DIR = Path(".data") / POINT_NAME

DEPTH_SENSORS = [0.1, 0.2, 0.3, 0.4]   # m below the river bed; the last sensor gives the bottom boundary condition
OFFSET = 0.0
NB_CELLS = 100
SIGMA_T = 0.1                          # accuracy of the temperature sensors, K

PRIORS = {
    "Prior_IntrinK": ((1e-15, 1e-10), 1e-12),   # (range, step): estimated, m2
    "Prior_n": (0.1, 0),                        # (value, 0): fixed
    "Prior_lambda_s": ((1.0, 5.0), 0.1),        # estimated, W/m/K
    "Prior_rhos_cs": (4e6, 0),                  # fixed, J/m3/K
    "Prior_q_s": (0.0, 0),                      # fixed, 1/s
}
ESTIMATED = ["IntrinK", "lambda_s"]
PARAMETERS = ["IntrinK", "n", "lambda_s", "rhos_cs", "q_s"]   # order of pyheatmy's Param

NB_CHAIN = 10
NB_ITER = 300
DISCARDED_FRACTION = 0.5
OUTLIER_IQR_FACTOR = 2.0   # outlier chains, as in DREAM (Vrugt et al., 2009)

SEED = 20261026
random.seed(SEED)
np.random.seed(SEED)

## 2. Reading the data set

The files have the format of the processed data of a real point. Dates are read as `yy/mm/dd HH:MM`, temperatures are converted from °C to kelvins, and the head difference is used as written, in metres.

`pyheatmy` takes two lists of dated measurements:
- `dH_measures`: `(date, (head difference, river temperature))`;
- `T_measures`: `(date, (T1, T2, T3, T4))`, where the last temperature is the bottom boundary condition and the three others are the measurements the model has to reproduce.

The model only uses the dates of `dH_measures`, so the two files must have the same dates; this is checked here.

In [ ]:
DATE_FORMAT = "%y/%m/%d %H:%M"

pressures = pd.read_csv(DATA_DIR / "processed_pressures.csv")
thermal = pd.read_csv(DATA_DIR / "processed_temperatures.csv")
assert (pressures.iloc[:, 0] == thermal.iloc[:, 0]).all(), "the two files must have the same dates"

dates = list(pd.to_datetime(pressures.iloc[:, 0], format=DATE_FORMAT))
head_difference = pressures.iloc[:, 1].to_numpy()
river_temperature = pressures.iloc[:, 2].to_numpy() + ZERO_CELSIUS
sensor_temperatures = thermal.iloc[:, 1:].to_numpy() + ZERO_CELSIUS

dH_measures = list(zip(dates, zip(head_difference, river_temperature)))
T_measures = list(zip(dates, sensor_temperatures))
print(f"{len(dates)} time steps, from {dates[0]} to {dates[-1]}, {sensor_temperatures.shape[1]} temperature sensors")

## 3. Column and priors

The values given to the layer are only placeholders for the estimated parameters: the chains start from points drawn from the priors. For the fixed parameters, the value of the prior is the one used.

In [ ]:
column = InversionColumn(
    river_bed=DEPTH_SENSORS[-1],
    depth_sensors=DEPTH_SENSORS,
    offset=OFFSET,
    dH_measures=dH_measures,
    T_measures=T_measures,
    nb_cells=NB_CELLS,
    rac=tempfile.mkdtemp(),
)

layer = Layer.from_dict({
    "name": "river bed",
    "zLow": DEPTH_SENSORS[-1],
    "IntrinK": 1e-13,
    "n": PRIORS["Prior_n"][0],
    "lambda_s": 3.0,
    "rhos_cs": PRIORS["Prior_rhos_cs"][0],
    "q_s": PRIORS["Prior_q_s"][0],
})
layer.set_priors_from_dict(PRIORS)
column.set_layers(layer)

for name, prior in zip(PARAMETERS, layer.Prior_list):
    print(f"{name:9s} scale: {prior.scale:6s} range in the sampling space: {prior.mcmc_range}")

## 4. MCMC

`compute_mcmc` runs DREAM with `NB_CHAIN` chains. Each iteration solves the direct model once per chain. The likelihood is Gaussian, with the variance of the measurement noise: in the `"no sigma"` mode, `sigma2` is fixed to `SIGMA_T**2` instead of being estimated.

Before the recorded iterations, `compute_mcmc` runs a burn-in phase of at most 25 iterations. This is not enough for the chains to gather where the posterior is (see the figure below), so the first `DISCARDED_FRACTION` of the recorded iterations is discarded as well.

A chain can also stay stuck in a region where the model fits the data much worse than the other chains do, for instance at a very low permeability. Such a chain is not a sample of the posterior. As in the original DREAM algorithm, a chain is set aside when its mean energy over the kept iterations exceeds the upper quartile of the chains' mean energies by more than `OUTLIER_IQR_FACTOR` interquartile ranges (the energy is minus the log-likelihood, up to a constant).

In [ ]:
column.compute_mcmc(
    typealgo="no sigma",
    sigma2=SIGMA_T**2,
    nb_iter=NB_ITER,
    nb_chain=NB_CHAIN,
    verbose=False,
)

# The recorded states are stored iteration by iteration, one per chain: the starting point, then NB_ITER iterations
history = np.array([list(state[0]) for state in column.get_all_params()]).reshape(NB_ITER + 1, NB_CHAIN, len(PARAMETERS))
first_kept = int(DISCARDED_FRACTION * NB_ITER)
energies = np.array(column.get_all_energy()).reshape(NB_ITER + 1, NB_CHAIN)

mean_energy = energies[first_kept:].mean(axis=0)
q1, q3 = np.quantile(mean_energy, [0.25, 0.75])
kept_chains = mean_energy <= q3 + OUTLIER_IQR_FACTOR * (q3 - q1)

print("Acceptance rate of each chain:", np.round(column.get_all_acceptance_ratio(), 3))
print("Mean energy of each chain over the kept iterations:", np.round(mean_energy, 1))
print(f"Chains set aside: {np.flatnonzero(~kept_chains).tolist()}")

In [ ]:
fig, axes = plt.subplots(len(ESTIMATED), 1, figsize=(10, 5), sharex=True)
for ax, name in zip(axes, ESTIMATED):
    values = history[:, :, PARAMETERS.index(name)]
    shown = np.log10(values) if name == "IntrinK" else values
    ax.plot(shown[:, kept_chains], linewidth=0.8)
    if not kept_chains.all():
        ax.plot(shown[:, ~kept_chains], color="grey", linestyle=":", linewidth=1.2)
    ax.axvline(first_kept, color="black", linestyle="--")
    ax.set_ylabel(r"$\log_{10} k$" if name == "IntrinK" else r"$\lambda_s$")
axes[0].set_title("Chains (one colour per chain, dotted grey: set aside); iterations left of the dashed line are discarded")
axes[-1].set_xlabel("iteration")
fig.tight_layout()

## 5. Comparison with the true parameters

`truth.json` is opened only now. The posterior is the set of states of the kept chains after the discarded part; the table gives its 5 %, 50 % and 95 % quantiles.

In [ ]:
truth = json.loads((DATA_DIR / "truth.json").read_text())
true_layer = truth["layers"][0]

posterior = history[first_kept:, kept_chains].reshape(-1, len(PARAMETERS))
QUANTILES = [0.05, 0.5, 0.95]
rows = []
for name in ESTIMATED:
    low, median, high = np.quantile(posterior[:, PARAMETERS.index(name)], QUANTILES)
    rows.append({"parameter": name, "true": true_layer[name], "5 %": low, "median": median, "95 %": high,
                 "true in [5 %, 95 %]": bool(low <= true_layer[name] <= high)})
pd.DataFrame(rows).set_index("parameter")

In [ ]:
fig, axes = plt.subplots(1, len(ESTIMATED), figsize=(10, 3.5))
for ax, name in zip(axes, ESTIMATED):
    ax.hist(posterior[:, PARAMETERS.index(name)], bins=40, density=True)
    ax.axvline(true_layer[name], color="red", label="true value")
    ax.set_xlabel(name)
axes[0].legend()
fig.tight_layout()

### Darcy velocity

The Darcy velocity is what the monitoring point is for. It is computed with the direct model for a sample of the posterior states, and for the true parameters.

In [ ]:
NB_REPLAYED = 50


def darcy_velocity(parameters):
    replay = Column(
        river_bed=DEPTH_SENSORS[-1], depth_sensors=DEPTH_SENSORS, offset=OFFSET,
        dH_measures=dH_measures, T_measures=T_measures, nb_cells=NB_CELLS, rac=tempfile.mkdtemp(),
    )
    replay.set_layers(Layer.from_dict({"name": "river bed", "zLow": DEPTH_SENSORS[-1], **parameters}))
    replay.compute_solve_transi(verbose=False)
    return replay.get_flows_solve().mean(), replay


rng = np.random.default_rng(SEED)
sample = posterior[rng.choice(len(posterior), size=NB_REPLAYED, replace=False)]
velocities = [darcy_velocity(dict(zip(PARAMETERS, state)))[0] for state in sample]
true_velocity, true_column = darcy_velocity({name: true_layer[name] for name in PARAMETERS})

low, median, high = np.quantile(velocities, QUANTILES)
print(f"Darcy velocity, true: {true_velocity:.4e} m/s")
print(f"Darcy velocity, posterior: median {median:.4e} m/s, 90 % interval [{low:.4e}, {high:.4e}] m/s")

### Temperatures

The temperatures computed with the median of the posterior are compared with the measurements. If the model and the noise level are right, the residuals have a standard deviation close to `SIGMA_T`.

In [ ]:
median_parameters = dict(zip(PARAMETERS, np.median(posterior, axis=0)))
_, median_column = darcy_velocity(median_parameters)
simulated = median_column.get_temperature_at_sensors()[1:-1]
measured = sensor_temperatures[:, :-1].T
residuals = measured - simulated
print(f"Standard deviation of the residuals: {residuals.std():.3f} K (noise of the sensors: {SIGMA_T} K)")

fig, ax = plt.subplots(figsize=(10, 4))
for depth, sim, meas in zip(DEPTH_SENSORS[:-1], simulated, measured):
    line, = ax.plot(dates, sim - ZERO_CELSIUS, label=f"{depth} m, median of the posterior")
    ax.plot(dates, meas - ZERO_CELSIUS, ".", markersize=2, color=line.get_color(), alpha=0.4)
ax.set_ylabel("T (°C)")
ax.set_title("Temperatures at the inner sensors: model (lines) and measurements (dots)")
ax.legend()
fig.tight_layout()

## 6. Reading the results

- **Convergence.** In the figure of section 4, the chains start spread over the prior range and gather around the same values within the first tens of iterations; after the dashed line they only move around these values. A chain set aside as an outlier stays far from the others with a much higher energy; if several chains are set aside, or if the kept ones do not gather, increase `NB_ITER`.
- **Parameters.** The true values of $k$ and $\lambda_s$ should lie within, or very close to, the 90 % intervals. The intervals are narrow because the data are long (two weeks, three sensors) and the noise is small.
- **Shape of the histograms.** With acceptance rates of a few percent, each chain moves only a few times over the kept iterations, so the posterior is represented by a few distinct values and its histograms are spiky. The intervals are therefore indicative; a longer run (larger `NB_ITER`) gives a smoother posterior.
- **Darcy velocity.** It follows $k$ directly; its interval should contain, or nearly contain, the true velocity.
- **Residuals.** A standard deviation close to `SIGMA_T` means the model explains the data down to the noise.

Things to try: change the seed or the noise in tutorial 1, shorten the data set, or free $n$ and $\rho_s c_s$ in `PRIORS` and see how much longer the chains need.